In [1]:
"Hello"

'Hello'

In [4]:
# Consumer: 병렬 처리 (노트북 2개에서 동시 실행)
from confluent_kafka import Consumer
import json
import time

config = {
    "bootstrap.servers": "kafka:9092",
    "group.id": "parallel-consumer-group",  # 같은 그룹 ID!
    "auto.offset.reset": "earliest",
}

consumer = Consumer(config)
consumer.subscribe(["api-events"])

print("Consumer 시작...")
count = 0
start = None
last_msg_time = None

empty_count = 0
max_empty = 3  # 1초 x 3번 = 3초 대기 후 종료
first_message = True

try:
    while True:
        # 첫 메시지는 10초 대기, 이후 1초 대기
        timeout = 10.0 if first_message else 1.0
        msg = consumer.poll(timeout)

        if msg is None:
            empty_count += 1
            if first_message:
                print("10초 동안 메시지 없음. 종료합니다.")
                break
            if empty_count >= max_empty:
                print("더 이상 메시지 없음. 종료합니다.")
                break
            continue

        if msg.error():
            continue

        empty_count = 0

        # 첫 메시지 수신 시 시간 측정 시작
        if first_message:
            start = time.time()
            first_message = False

        last_msg_time = time.time()
        count += 1

        if count % 100000 == 0:
            print(f"  {count:,}건 처리중 (파티션 {msg.partition()})")

except KeyboardInterrupt:
    pass
finally:
    consumer.close()

# 결과 출력 (대기 시간 제외, 실제 처리 시간만 측정)
print(f"\n{'=' * 50}")
if count > 0 and start and last_msg_time:
    elapsed = last_msg_time - start
    print(f"총 수신: {count:,}건")
    print(f"소요 시간: {elapsed:.2f}초 (대기 시간 제외)")
    print(f"처리량: {count / elapsed:,.0f} records/sec")
else:
    print("수신된 메시지 없음")

Consumer 시작...
  100,000건 처리중 (파티션 2)
  200,000건 처리중 (파티션 3)
  300,000건 처리중 (파티션 3)
  400,000건 처리중 (파티션 3)
  500,000건 처리중 (파티션 2)
더 이상 메시지 없음. 종료합니다.

총 수신: 516,245건
소요 시간: 3.26초 (대기 시간 제외)
처리량: 158,291 records/sec
